# Thesis Visualizations — Dataset Overview

Produces publication-quality figures for the curated Celeb-DF++ subset.

| Figure | Content |
|--------|---------|
| **Fig. 2** | Class (real/fake) and forgery-method distribution |
| **Fig. 3** | Sample frame grid — real (top) vs fake (bottom) |

Figures are saved to `../outputs/figures/`. All data is embedded from the Stage-1 manifest;
if the manifest CSVs are available locally they will be loaded automatically.

**Datasets:** Pilot (200 videos, 100 R + 100 F) · Final (800 videos, 400 R + 400 F) · Combined (1000 videos, 500 R + 500 F)

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import to_rgba
import warnings
warnings.filterwarnings('ignore')

# ── Output directory ───────────────────────────────────────────────────────────
FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Thesis-grade rcParams ──────────────────────────────────────────────────────
mpl.rcParams.update({
    'font.family':          'sans-serif',
    'font.sans-serif':      ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':            10,
    'axes.titlesize':       11,
    'axes.titleweight':     'bold',
    'axes.labelsize':       10,
    'axes.labelcolor':      '#1A1A1A',
    'axes.linewidth':       0.8,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'axes.edgecolor':       '#AAAAAA',
    'axes.grid':            True,
    'axes.grid.axis':       'x',
    'grid.color':           '#E8E8E8',
    'grid.linewidth':       0.6,
    'xtick.color':          '#444444',
    'ytick.color':          '#444444',
    'xtick.labelsize':      9,
    'ytick.labelsize':      9,
    'legend.fontsize':      9,
    'legend.framealpha':    0.9,
    'legend.edgecolor':     '#CCCCCC',
    'figure.dpi':           150,
    'savefig.dpi':          300,
    'savefig.bbox':         'tight',
    'savefig.facecolor':    'white',
    'text.color':           '#1A1A1A',
})

# ── Colour palette ─────────────────────────────────────────────────────────────
REAL_COLOR   = '#2D6A4F'   # forest green — authentic
FAKE_COLOR   = '#9B2226'   # deep red     — manipulated (overview only)

FAMILY_COLORS = {
    'FaceReenact': '#C1440E',
    'FaceSwap':    '#5C4B8A',
    'TalkingFace': '#1A6FA0',
}

# Per-method colours: lighter tint of the family colour
def _method_color(family: str, alpha: float = 0.78) -> str:
    base = to_rgba(FAMILY_COLORS[family])
    white = np.array([1, 1, 1, 1])
    blended = np.array(base) * alpha + white * (1 - alpha)
    return tuple(blended)

print('Setup complete. Figures will be saved to:', FIGURES_DIR.resolve())

In [ ]:
# ── Dataset distributions (from Stage-1 manifest outputs) ─────────────────────
# These are the exact counts produced by celeb_dfpp_dataset_overview.ipynb.
# If manifest CSVs are available locally, they will be loaded automatically.

PILOT_FAKE = {
    'FaceReenact': {
        'DaGAN': 5, 'FSRT': 5, 'HyperReenact': 5, 'LIA': 5,
        'LivePortrait': 5, 'MCNET': 5, 'TPSMM': 4,
    },
    'FaceSwap': {
        'BlendFace': 5, 'Celeb-DF-v2': 4, 'GHOST': 4, 'HifiFace': 4,
        'InSwapper': 4, 'MobileFaceSwap': 4, 'SimSwap': 4, 'UniFace': 4,
    },
    'TalkingFace': {
        'AniTalker': 5, 'EDTalk': 5, 'EchoMimic': 5, 'FLOAT': 5,
        'IP_LAP': 5, 'Real3DPortrait': 4, 'SadTalker': 4,
    },
}
PILOT_REAL = 100

FINAL_FAKE = {
    'FaceReenact': {
        'DaGAN': 20, 'FSRT': 19, 'HyperReenact': 19, 'LIA': 19,
        'LivePortrait': 19, 'MCNET': 19, 'TPSMM': 19,
    },
    'FaceSwap': {
        'BlendFace': 17, 'Celeb-DF-v2': 17, 'GHOST': 17, 'HifiFace': 17,
        'InSwapper': 17, 'MobileFaceSwap': 16, 'SimSwap': 16, 'UniFace': 16,
    },
    'TalkingFace': {
        'AniTalker': 19, 'EDTalk': 19, 'EchoMimic': 19, 'FLOAT': 19,
        'IP_LAP': 19, 'Real3DPortrait': 19, 'SadTalker': 19,
    },
}
FINAL_REAL = 400


def _build_method_df(fake_dict: dict, real_count: int, dataset_name: str) -> pd.DataFrame:
    """Flatten nested family→method dict into a tidy DataFrame, prepend Real row."""
    rows = []
    for family, methods in fake_dict.items():
        for method, count in methods.items():
            rows.append({'family': family, 'method': method, 'count': count,
                         'label': 'Fake', 'dataset': dataset_name})
    real_row = {'family': 'Real', 'method': 'Real', 'count': real_count,
                'label': 'Real', 'dataset': dataset_name}
    return pd.DataFrame([real_row] + rows)


def _combine_fake_dicts(d1: dict, d2: dict) -> dict:
    """Element-wise sum of two family→method count dicts."""
    combined = {}
    for family in set(d1) | set(d2):
        methods = set(d1.get(family, {})) | set(d2.get(family, {}))
        combined[family] = {
            m: d1.get(family, {}).get(m, 0) + d2.get(family, {}).get(m, 0)
            for m in methods
        }
    return combined


COMBINED_FAKE = _combine_fake_dicts(PILOT_FAKE, FINAL_FAKE)
COMBINED_REAL = PILOT_REAL + FINAL_REAL

pilot_df    = _build_method_df(PILOT_FAKE,    PILOT_REAL,    'Pilot')
final_df    = _build_method_df(FINAL_FAKE,    FINAL_REAL,    'Final')
combined_df = _build_method_df(COMBINED_FAKE, COMBINED_REAL, 'Combined')

# Summary
for name, df, n_real, fake_d in [
    ('Pilot',    pilot_df,    PILOT_REAL,    PILOT_FAKE),
    ('Final',    final_df,    FINAL_REAL,    FINAL_FAKE),
    ('Combined', combined_df, COMBINED_REAL, COMBINED_FAKE),
]:
    n_fake = sum(v for fam in fake_d.values() for v in fam.values())
    print(f'{name:>10s}: {n_real + n_fake:4d} videos  |  '
          f'Real: {n_real:4d}  Fake: {n_fake:4d}  '
          f'Methods: {sum(len(v) for v in fake_d.values())}')

## Figure 2 · Class and Method Distribution

Three views: (1) Pilot + Final **combined**, (2) **Pilot** only, (3) **Final** only.

In [ ]:
# ─── Shared helper: draw one distribution panel ───────────────────────────────

def _bar_color(row: pd.Series) -> str:
    if row['label'] == 'Real':
        return REAL_COLOR
    return FAMILY_COLORS[row['family']]


def draw_class_balance(ax, n_real: int, n_fake: int, title: str = '') -> None:
    """Left panel: stacked horizontal bar showing real/fake split with counts."""
    total = n_real + n_fake
    pct_r = n_real / total * 100
    pct_f = n_fake / total * 100

    ax.barh([''], [pct_r], color=REAL_COLOR, height=0.45, label=f'Real ({n_real})')
    ax.barh([''], [pct_f], left=[pct_r], color=FAKE_COLOR,
            height=0.45, label=f'Fake ({n_fake})')

    ax.text(pct_r / 2, 0, f'{n_real}\n({pct_r:.0f}%)',
            ha='center', va='center', fontsize=10, fontweight='bold',
            color='white')
    ax.text(pct_r + pct_f / 2, 0, f'{n_fake}\n({pct_f:.0f}%)',
            ha='center', va='center', fontsize=10, fontweight='bold',
            color='white')

    ax.set_xlim(0, 100)
    ax.set_xlabel('Proportion (%)')
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.grid(False)
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
    if title:
        ax.set_title(title, pad=8)


def draw_family_donut(ax, fake_dict: dict, n_real: int, title: str = '') -> None:
    """Left panel (alternative): nested donut — inner=real/fake, outer=family."""
    n_fake = sum(v for fam in fake_dict.values() for v in fam.values())
    total  = n_real + n_fake

    # Inner ring: Real vs Fake
    inner_sizes  = [n_real, n_fake]
    inner_colors = [REAL_COLOR, '#DDDDDD']
    ax.pie(inner_sizes, colors=inner_colors, radius=0.55,
           wedgeprops=dict(width=0.28, edgecolor='white', linewidth=1.5),
           startangle=90)

    # Outer ring: Real + families
    fam_totals = {fam: sum(m.values()) for fam, m in fake_dict.items()}
    outer_labels = ['Real'] + list(fam_totals.keys())
    outer_sizes  = [n_real] + list(fam_totals.values())
    outer_colors = [REAL_COLOR] + [FAMILY_COLORS[f] for f in fam_totals]
    ax.pie(outer_sizes, colors=outer_colors, radius=0.90,
           wedgeprops=dict(width=0.30, edgecolor='white', linewidth=1.5),
           startangle=90)

    # Centre label
    ax.text(0, 0, f'{total}\nvideos', ha='center', va='center',
            fontsize=11, fontweight='bold', color='#1A1A1A')

    # Legend
    handles = [
        mpatches.Patch(color=REAL_COLOR, label=f'Real ({n_real})')
    ] + [
        mpatches.Patch(color=FAMILY_COLORS[f], label=f'{f} ({fam_totals[f]})')
        for f in fam_totals
    ]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.22),
              ncol=2, framealpha=0.9, fontsize=9)
    if title:
        ax.set_title(title, pad=12)


def draw_method_bars(ax, method_df: pd.DataFrame, title: str = '',
                     show_family_labels: bool = True) -> None:
    """Right panel: horizontal bar chart of all methods, grouped by family."""
    # Order: Real first, then family groups
    family_order = ['FaceReenact', 'FaceSwap', 'TalkingFace']
    ordered_rows = []
    for family in family_order:
        fam_rows = method_df[method_df['family'] == family].copy()
        fam_rows = fam_rows.sort_values('method')
        ordered_rows.append(fam_rows)
    real_row = method_df[method_df['label'] == 'Real']
    plot_df  = pd.concat([real_row] + ordered_rows, ignore_index=True)

    y_labels = plot_df['method'].tolist()
    y_pos    = np.arange(len(y_labels))
    colors   = [_bar_color(row) for _, row in plot_df.iterrows()]
    counts   = plot_df['count'].tolist()

    bars = ax.barh(y_pos, counts, color=colors, height=0.65,
                   edgecolor='white', linewidth=0.5)

    # Annotate counts
    x_max = max(counts) * 1.15
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + x_max * 0.012, bar.get_y() + bar.get_height() / 2,
                str(count), va='center', ha='left', fontsize=8.5, color='#333333')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlim(0, x_max)
    ax.set_xlabel('Number of videos')
    ax.invert_yaxis()
    ax.grid(True, axis='x', color='#E8E8E8', linewidth=0.6)
    ax.grid(False, axis='y')
    ax.spines['left'].set_visible(False)

    if show_family_labels:
        # Draw family dividers and labels
        cumulative = 1  # skip 'Real'
        for family in family_order:
            n_methods = len(method_df[method_df['family'] == family])
            if cumulative > 1:
                ax.axhline(cumulative - 0.5, color='#CCCCCC', linewidth=0.8, linestyle='--')
            mid = cumulative + n_methods / 2 - 0.5
            ax.text(-x_max * 0.01, mid, family,
                    ha='right', va='center', fontsize=8, fontweight='bold',
                    color=FAMILY_COLORS[family],
                    transform=ax.get_yaxis_transform())
            cumulative += n_methods

        ax.axhline(0.5, color='#CCCCCC', linewidth=0.8, linestyle='--')

    if title:
        ax.set_title(title, pad=8)

    # Legend for families
    handles = [mpatches.Patch(color=REAL_COLOR, label='Real')] + [
        mpatches.Patch(color=FAMILY_COLORS[f], label=f) for f in family_order
    ]
    ax.legend(handles=handles, loc='lower right', fontsize=8.5,
              framealpha=0.9, title='Category', title_fontsize=8)


print('Helper drawing functions defined.')

In [ ]:
# ─── Fig. 2 — Combined (Pilot + Final) ─────────────────────────────────────────

n_real_comb = COMBINED_REAL
n_fake_comb = sum(v for fam in COMBINED_FAKE.values() for v in fam.values())

fig = plt.figure(figsize=(13, 6.5))
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.6], wspace=0.38)

ax_donut = fig.add_subplot(gs[0])
ax_bars  = fig.add_subplot(gs[1])

draw_family_donut(
    ax_donut, COMBINED_FAKE, n_real_comb,
    title='(a) Class and category distribution'
)
draw_method_bars(
    ax_bars, combined_df,
    title='(b) Video count per forgery method'
)

fig.suptitle(
    'Fig. 2.  Class and method distribution of the curated subset  '
    '(Pilot + Final combined, N = 1 000)',
    fontsize=11, fontweight='bold', y=1.01, ha='center'
)

out = FIGURES_DIR / 'fig2_combined_distribution.pdf'
fig.savefig(out, bbox_inches='tight')
fig.savefig(str(out).replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# ─── Fig. 2 — Pilot subset ─────────────────────────────────────────────────────

n_fake_pilot = sum(v for fam in PILOT_FAKE.values() for v in fam.values())

fig = plt.figure(figsize=(13, 6.5))
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.6], wspace=0.38)

ax_donut = fig.add_subplot(gs[0])
ax_bars  = fig.add_subplot(gs[1])

draw_family_donut(
    ax_donut, PILOT_FAKE, PILOT_REAL,
    title='(a) Class and category distribution'
)
draw_method_bars(
    ax_bars, pilot_df,
    title='(b) Video count per forgery method'
)

fig.suptitle(
    'Fig. 2a.  Class and method distribution — Pilot subset  '
    f'(N = {PILOT_REAL + n_fake_pilot})',
    fontsize=11, fontweight='bold', y=1.01, ha='center'
)

out = FIGURES_DIR / 'fig2a_pilot_distribution.pdf'
fig.savefig(out, bbox_inches='tight')
fig.savefig(str(out).replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# ─── Fig. 2 — Final subset ─────────────────────────────────────────────────────

n_fake_final = sum(v for fam in FINAL_FAKE.values() for v in fam.values())

fig = plt.figure(figsize=(13, 6.5))
gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.6], wspace=0.38)

ax_donut = fig.add_subplot(gs[0])
ax_bars  = fig.add_subplot(gs[1])

draw_family_donut(
    ax_donut, FINAL_FAKE, FINAL_REAL,
    title='(a) Class and category distribution'
)
draw_method_bars(
    ax_bars, final_df,
    title='(b) Video count per forgery method'
)

fig.suptitle(
    'Fig. 2b.  Class and method distribution — Final subset  '
    f'(N = {FINAL_REAL + n_fake_final})',
    fontsize=11, fontweight='bold', y=1.01, ha='center'
)

out = FIGURES_DIR / 'fig2b_final_distribution.pdf'
fig.savefig(out, bbox_inches='tight')
fig.savefig(str(out).replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# ─── Fig. 2 — Side-by-side comparison: Pilot vs Final ──────────────────────────
#
# A 2×2 grid comparing Pilot (left column) with Final (right column).
# Row 1: donut charts; Row 2: method bar charts.

fig = plt.figure(figsize=(15, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, wspace=0.40, hspace=0.45)

ax_p_donut = fig.add_subplot(gs[0, 0])
ax_f_donut = fig.add_subplot(gs[0, 1])
ax_p_bars  = fig.add_subplot(gs[1, 0])
ax_f_bars  = fig.add_subplot(gs[1, 1])

draw_family_donut(ax_p_donut, PILOT_FAKE, PILOT_REAL,
                  title='Pilot — class & category')
draw_family_donut(ax_f_donut, FINAL_FAKE, FINAL_REAL,
                  title='Final — class & category')
draw_method_bars(ax_p_bars, pilot_df,
                 title='Pilot — videos per method')
draw_method_bars(ax_f_bars, final_df,
                 title='Final — videos per method')

fig.suptitle(
    'Fig. 2.  Class and method distribution: Pilot (N = 200) vs Final (N = 800)',
    fontsize=12, fontweight='bold', y=1.01, ha='center'
)

out = FIGURES_DIR / 'fig2_pilot_vs_final_comparison.pdf'
fig.savefig(out, bbox_inches='tight')
fig.savefig(str(out).replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Figure 3 · Sample Frame Grid

2 rows × 4 columns showing representative frames.  
- **Top row**: real videos (from Celeb-real and YouTube-real sources)  
- **Bottom row**: one fake per forgery family, plus a fourth variant  

The cell below tries to load actual video frames from the manifest CSVs. If the video files are not available on this machine, it falls back to informative placeholder tiles (consistent colour-coded by family).

In [ ]:
# ─── Frame extraction utilities ────────────────────────────────────────────────
import cv2

MANIFEST_CANDIDATES = [
    Path('../metadata/pilot_manifest.csv'),
    Path('../metadata/final_manifest.csv'),
    Path('../exports/pilot/csv/pilot_manifest_export.csv'),
    Path('../exports/final/csv/final_manifest_export.csv'),
]

DATASET_ROOTS = [
    Path('/home/n-salikhova/datasets/extracted/Celeb-DF-v3'),
    Path.home() / 'datasets' / 'Celeb-DF-v3',
]


def _load_manifest() -> pd.DataFrame | None:
    frames = []
    for p in MANIFEST_CANDIDATES:
        if p.exists():
            frames.append(pd.read_csv(p))
    if not frames:
        return None
    df = pd.concat(frames, ignore_index=True).drop_duplicates('video_id')
    return df


def _resolve_path(raw: str) -> Path | None:
    p = Path(raw)
    if p.exists():
        return p
    for root in DATASET_ROOTS:
        candidate = root / p.name
        if candidate.exists():
            return candidate
        # relative_path stored in manifest (e.g. Celeb-real/id2_0008.mp4)
        for part in p.parts:
            if part.endswith('.mp4'):
                for dr in DATASET_ROOTS:
                    cand2 = dr / Path(raw).relative_to(Path(raw).parts[0]) if len(Path(raw).parts) > 1 else None
                    if cand2 and cand2.exists():
                        return cand2
    return None


def extract_frame(video_path: str, frame_idx: int = None) -> np.ndarray | None:
    """Extract a single frame from a video file. Returns BGR array or None."""
    resolved = _resolve_path(str(video_path))
    if resolved is None:
        return None
    cap = cv2.VideoCapture(str(resolved))
    if not cap.isOpened():
        return None
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_idx is None:
        frame_idx = max(1, n_frames // 3)
    cap.set(cv2.CAP_PROP_POS_FRAMES, min(frame_idx, n_frames - 1))
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None


def _placeholder_tile(label: str, method: str, family: str | None,
                       size: tuple = (224, 224)) -> np.ndarray:
    """Colour-coded placeholder tile when video file is unavailable."""
    h, w = size
    if family and family in FAMILY_COLORS:
        r, g, b, _ = to_rgba(FAMILY_COLORS[family])
    elif label == 'real':
        r, g, b, _ = to_rgba(REAL_COLOR)
    else:
        r, g, b = 0.4, 0.4, 0.4
    img = np.full((h, w, 3), [int(r * 200 + 55), int(g * 200 + 55), int(b * 200 + 55)],
                  dtype=np.uint8)
    return img


print('Frame extraction utilities loaded.')

In [ ]:
# ─── Fig. 3 — Sample frame grid ───────────────────────────────────────────────
#
# Layout: 2 rows × 4 columns
# Row 0 (top):    4 real frames
# Row 1 (bottom): 4 fake frames, one per distinct forgery family/method

# Desired grid entries:
# Real row: 4 different real videos (mixed Celeb-real, YouTube-real)
# Fake row: FaceReenact·DaGAN | FaceSwap·BlendFace | TalkingFace·EchoMimic | FaceSwap·HifiFace

GRID_SPEC = [
    # (row, label, family, method, subtitle)
    (0, 'real',   None,          None,              'Real'),
    (0, 'real',   None,          None,              'Real'),
    (0, 'real',   None,          None,              'Real'),
    (0, 'real',   None,          None,              'Real'),
    (1, 'fake',  'FaceReenact', 'DaGAN',            'FaceReenact\n(DaGAN)'),
    (1, 'fake',  'FaceSwap',    'BlendFace',        'FaceSwap\n(BlendFace)'),
    (1, 'fake',  'TalkingFace', 'EchoMimic',        'TalkingFace\n(EchoMimic)'),
    (1, 'fake',  'FaceSwap',    'HifiFace',         'FaceSwap\n(HifiFace)'),
]


def _pick_video(manifest: pd.DataFrame | None, label: str,
                family: str | None, method: str | None,
                used_ids: set, seed_offset: int = 0) -> tuple[pd.Series | None, int]:
    """Pick one unused video row from the manifest matching the criteria."""
    if manifest is None:
        return None, seed_offset
    mask = manifest['label'] == label
    if family:
        mask &= manifest['manipulation_family'] == family
    if method:
        mask &= manifest['manipulation_type'] == method
    mask &= ~manifest['video_id'].isin(used_ids)
    pool = manifest[mask]
    if pool.empty:
        return None, seed_offset
    row = pool.sample(1, random_state=42 + seed_offset).iloc[0]
    return row, seed_offset + 1


manifest = _load_manifest()
if manifest is not None:
    print(f'Manifest loaded: {len(manifest)} rows')
else:
    print('Manifest not found locally — placeholder tiles will be used.')

# Build frame list
frames_data = []  # list of (rgb_array, subtitle, label, family)
used_ids = set()
seed_ctr = 0

real_count = 0
for row_idx, label, family, method, subtitle in GRID_SPEC:
    frame = None
    row_meta, seed_ctr = _pick_video(manifest, label, family, method, used_ids, seed_ctr)

    if row_meta is not None:
        used_ids.add(row_meta['video_id'])
        frame = extract_frame(row_meta.get('path', ''), frame_idx=None)
        if frame is not None:
            # Crop centre square
            h, w = frame.shape[:2]
            s = min(h, w)
            y0, x0 = (h - s) // 2, (w - s) // 2
            frame = frame[y0:y0 + s, x0:x0 + s]
            frame = cv2.resize(frame, (224, 224), interpolation=cv2.INTER_AREA)

    if frame is None:
        frame = _placeholder_tile(label, subtitle, family)

    frames_data.append((frame, subtitle, label, family))

# ── Draw the grid ─────────────────────────────────────────────────────────────
N_COLS = 4
N_ROWS = 2
PAD    = 0.04   # fraction of image size for border

fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=(N_COLS * 2.6, N_ROWS * 2.9),
    gridspec_kw={'hspace': 0.28, 'wspace': 0.08},
)

ROW_LABELS = ['Real', 'Fake']
ROW_LABEL_COLORS = [REAL_COLOR, FAKE_COLOR]

for idx, (frame, subtitle, label, family) in enumerate(frames_data):
    r, c = divmod(idx, N_COLS)
    ax = axes[r, c]
    ax.imshow(frame, interpolation='bilinear')
    ax.axis('off')

    # Subtitle below
    col = FAMILY_COLORS.get(family, REAL_COLOR) if label == 'fake' else REAL_COLOR
    ax.set_xlabel(subtitle, fontsize=8, labelpad=4, color=col, fontweight='bold')
    ax.xaxis.set_label_position('bottom')

    # Coloured border
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(col)
        spine.set_linewidth(2.5)

# Row annotation (left side)
for r_idx, (row_label, row_color) in enumerate(zip(ROW_LABELS, ROW_LABEL_COLORS)):
    axes[r_idx, 0].annotate(
        row_label,
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-18, 0), textcoords='offset points',
        ha='right', va='center',
        fontsize=10, fontweight='bold', color=row_color,
        rotation=90,
    )

fig.suptitle(
    'Fig. 3.  Sample frames from the curated subset\n'
    '(top row: real; bottom row: fake — one per forgery type)',
    fontsize=10.5, fontweight='bold', y=1.03, ha='center'
)

# Source annotation
src_note = 'Frames from Celeb-DF-v3. Placeholder tiles shown when video files are unavailable locally.'
fig.text(0.5, -0.01, src_note, ha='center', fontsize=7.5, color='#777777', style='italic')

out = FIGURES_DIR / 'fig3_sample_frames.pdf'
fig.savefig(out, bbox_inches='tight')
fig.savefig(str(out).replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# ─── Summary of saved files ────────────────────────────────────────────────────
print('Saved figures:')
for f in sorted(FIGURES_DIR.glob('fig*.*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<50s}  {size_kb:>7.1f} KB')